In [ ]:
import pandas as pd
import numpy as np
import pickle
from time import time

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.preprocessing import StandardScaler

# ── Models ──────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    BaggingClassifier,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
print("Loading Dataset...")
data = pd.read_csv("heartdata.csv")
print("Heart Dataset successfully loaded")


In [ ]:
print("\n===== BASIC INFO =====")
print("\nMissing Values:\n", data.isnull().sum())
print("\nClass Distribution:\n", data["target"].value_counts())
print("\nCorrelation with Target:\n")
print(data.corr()["target"].sort_values(ascending=False))
print("\nColumns:", list(data.columns))

In [ ]:
X = data.drop("target", axis=1)   # Independent Variables
y = data["target"]                 # Dependent Variable (target)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
models = {
    "Logistic Regression":   LogisticRegression(max_iter=1000),
    "SVM":                   SVC(probability=True),
    "Random Forest":         RandomForestClassifier(),
    "Decision Tree":         DecisionTreeClassifier(),
    "KNN":                   KNeighborsClassifier(),
    "Naive Bayes":           GaussianNB(),
    "Gradient Boosting":     GradientBoostingClassifier(),
    "AdaBoost":              AdaBoostClassifier(),
    "Bagging Classifier":    BaggingClassifier(),
    "MLP Classifier":        MLPClassifier(max_iter=500),
    "XGBoost":               XGBClassifier(use_label_encoder=False, eval_metric="logloss"),
}
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

In [ ]:
print("\n===== MODEL EVALUATION (5-Fold Cross-Validation) =====\n")

results      = []
model_store  = {}   # keep fitted models for later

for name, model in models.items():

    start_time = time()

    # Cross-validate on the scaled training data
    cv_scores = cross_validate(
        model,
        X_train_scaled, y_train,
        cv=5,
        scoring=scoring,
        return_train_score=False,
    )

    elapsed_time = time() - start_time

    # Average each metric across the 5 folds
    accuracy  = np.mean(cv_scores["test_accuracy"])
    precision = np.mean(cv_scores["test_precision"])
    recall    = np.mean(cv_scores["test_recall"])
    f1        = np.mean(cv_scores["test_f1"])
    auc       = np.mean(cv_scores["test_roc_auc"])

    results.append({
        "Model":      name,
        "Accuracy":   accuracy,
        "Precision":  precision,
        "Recall":     recall,
        "F1-Score":   f1,
        "AUC":        auc,
        "Time (s)":   elapsed_time,
        "_model_obj": model,   # hidden column, removed before display
    })

    print(f"{name} completed in {elapsed_time:.4f} seconds.")
    print(f"  Accuracy : {accuracy:.3f}")
    print(f"  Precision: {precision:.3f}")
    print(f"  Recall   : {recall:.3f}")
    print(f"  F1-Score : {f1:.3f}")
    print(f"  AUC      : {auc:.3f}")
    print("-" * 35)


In [ ]:
results_df = pd.DataFrame(results)
display_df = results_df.drop(columns=["_model_obj"])

print("\n===== FULL COMPARISON TABLE =====\n")
print(display_df.sort_values("Recall", ascending=False).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sorted_df = results_df.sort_values("Accuracy", ascending=True)

plt.figure(figsize=(12,6))
sns.barplot(x="Accuracy", y="Model", data=sorted_df, palette="viridis")
plt.title("Accuracy Comparison of Models")
plt.show()

In [ ]:
sorted_df = results_df.sort_values("F1-Score", ascending=True)

plt.figure(figsize=(12,6))
sns.barplot(x="F1-Score", y="Model", data=sorted_df, palette="viridis", hue="Model", legend=False)
plt.title("F1-Score Comparison of Models")
plt.show()

In [ ]:
sorted_df = results_df.sort_values("Accuracy", ascending=False)

sns.set(rc={'figure.figsize':(15,8)})
plt.xlabel("Algorithms")
plt.ylabel("Accuracy score")

sns.barplot(x=sorted_df["Model"], y=sorted_df["Accuracy"])